### 1. Objective and Methodology
* **Goal:** Use SHAP (SHapley Additive exPlanations) values to interpret the trained XGBoost model (`best_xgb`) and identify which network features drive its predictions.
* **Implementation:** Sampled 2,000 random test instances, converted normalized features back to original unscaled units using inverse transformation, and used `shap.TreeExplainer` to calculate Shapley values.

---

### 2. Key Interpretability Insights
* **Global Importance:** Summary and bar plots show which network flow metrics (like duration, packet inter-arrival times, and rates) have the greatest overall impact on classification.
* **Directional Effects:** The beeswarm plot illustrates how specific high or low feature values push predictions toward benign or malicious labels.
* **Local Explainability:** The waterfall plot for a specific detected attack details the exact additive contribution of each feature to that prediction.

---

### 3. Artifacts and Exports
* **Visuals:** Generated and saved the following plots to `../figures/`:
  * `shap_summary.png` (Global beeswarm plot)
  * `shap_bar.png` (Global feature importance bar chart)
  * `shap_waterfall_attack.png` (Local instance waterfall plot)

In [5]:
import sys
sys.path.append('../src')
import joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import load
from xgboost import XGBClassifier

DATASET = 'CICIDS2017_output'
Xtr, ytr, Xte, yte, feats = load('../data/' + DATASET)

spw = (ytr == 0).sum() / (ytr == 1).sum()
xgb = XGBClassifier(tree_method='hist', eval_metric='logloss',
                    scale_pos_weight=spw, n_jobs=-1, random_state=42).fit(Xtr, ytr)

joblib.dump(xgb, '../exports/xgb_default_' + DATASET + '.joblib')   # so you never need to retrain
print(Xtr.shape, Xte.shape)

(1764558, 30) (756240, 30)


In [7]:
import shap
scaler = joblib.load('../data/' + DATASET + '/scaler.joblib')
idx = np.random.RandomState(42).choice(len(Xte), 2000, replace=False)
Xs = pd.DataFrame(Xte[idx], columns=feats)
Xs_raw = pd.DataFrame(scaler.inverse_transform(Xte[idx]), columns=feats)
explainer = shap.TreeExplainer(xgb)
sv = explainer(Xs)

shap.summary_plot(sv.values, Xs_raw, show=False)
plt.savefig('../figures/shap_summary.png', dpi=200, bbox_inches='tight'); plt.close()
shap.summary_plot(sv.values, Xs_raw, plot_type='bar', show=False)
plt.savefig('../figures/shap_bar.png', dpi=200, bbox_inches='tight'); plt.close()

pred = xgb.predict(Xs)
i = int(np.where((pred == 1) & (yte[idx] == 1))[0][0])
shap.plots.waterfall(sv[i], show=False)
plt.savefig('../figures/shap_waterfall_attack.png', dpi=200, bbox_inches='tight'); plt.close()
print('saved 3 figures; explained flow index:', i)

saved 3 figures; explained flow index: 2


---

In [12]:
j = int(np.where((pred == 0) & (yte[idx] == 0))[0][0])      # a correctly classified benign flow
for name, k in [('attack', i), ('benign', j)]:
    ex = shap.Explanation(values=sv.values[k], base_values=sv.base_values[k],
                          data=Xs_raw.iloc[k].values, feature_names=feats)
    shap.plots.waterfall(ex, show=False)
    plt.savefig('../figures/shap_waterfall_' + name + '_raw_' + DATASET + '.png', dpi=200, bbox_inches='tight')
    plt.close()

In [13]:
from scipy.stats import spearmanr
imp = pd.read_csv('../data/' + DATASET + '/feature_importance.csv', index_col=0).iloc[:, 0].loc[feats]
mean_abs = pd.Series(np.abs(sv.values).mean(0), index=feats)
print(spearmanr(mean_abs.values, imp.values))
display(pd.DataFrame({'SHAP': mean_abs, 'RF importance': imp}).sort_values('SHAP', ascending=False).head(10).round(4))

SignificanceResult(statistic=np.float64(0.28898776418242494), pvalue=np.float64(0.12141718912490895))


,SHAP,RF importance
Destination Port,2.0720,0.1001
Init_Win_bytes_forward,1.9817,0.0351
Init_Win_bytes_backward,1.9492,0.0553
Packet Length Variance,1.7671,0.0271
Fwd Packet Length Max,0.8430,0.0484
Bwd Packet Length Max,0.8325,0.0288
Fwd Header Length,0.6019,0.0251
Flow IAT Min,0.5738,0.0169
Bwd Packets/s,0.5282,0.0399
Packet Length Mean,0.5150,0.0296


In [14]:
from common import evaluate
from xgboost import XGBClassifier
keep = [k for k, n in enumerate(feats) if n != 'Destination Port']
m2 = XGBClassifier(**xgb.get_params()).fit(Xtr[:, keep], ytr)

r1, _ = evaluate('XGB, all 30 features', lambda X: xgb.predict_proba(X)[:, 1], Xte, yte)
r2, _ = evaluate('XGB, no Destination Port', lambda X: m2.predict_proba(X[:, keep])[:, 1], Xte, yte)
ablation = pd.DataFrame([r1, r2]).round(4)
ablation.to_csv('../results/' + DATASET + '_ablation_port.csv', index=False)
display(ablation)

,model,accuracy,precision,recall,f1,roc_auc,fpr,fnr,infer_us
0,"XGB, all 30 features",0.9991,0.9947,0.9998,0.9973,1.0,0.0011,0.0002,0.6507
1,"XGB, no Destination Port",0.9988,0.9936,0.9995,0.9966,1.0,0.0013,0.0005,0.5990


In [15]:
import sklearn, xgboost, os
os.makedirs('../exports', exist_ok=True)
bundle = dict(model=xgb, scaler=scaler, features=feats, threshold=0.5,
              label_map={0: 'BENIGN', 1: 'ATTACK'}, dataset=DATASET,
              versions=dict(sklearn=sklearn.__version__, xgboost=xgboost.__version__))
joblib.dump(bundle, '../exports/final_model.joblib')

# sanity test: reload and predict on 5 raw flows
b = joblib.load('../exports/final_model.joblib')
raw = pd.DataFrame(b['scaler'].inverse_transform(Xte[:5]), columns=b['features'])
print(b['model'].predict_proba(b['scaler'].transform(raw[b['features']]))[:, 1].round(4), yte[:5])

[0. 0. 0. 1. 0.] [0 0 0 1 0]
